In [36]:
import pandas as pd
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors
import numpy as np
import plotly.express as px


In [37]:

# CONFIG
train_csv = 'train.csv'
taxonomy_csv = 'taxonomy.csv'
output_html = 'birdclef2025_map.html'

In [38]:
# Load data
train_df = pd.read_csv(train_csv)
taxonomy_df = pd.read_csv(taxonomy_csv)
taxonomy_df = taxonomy_df.rename(columns={taxonomy_df.columns[0]: 'id',
                                        taxonomy_df.columns[3]: 'name'})

In [39]:
# Merge name with train.csv
data = train_df[[train_df.columns[0], train_df.columns[3], train_df.columns[7], train_df.columns[8]]]
data.columns = ['id', 'filename', 'lat', 'lon']
data = data.merge(taxonomy_df[['id', 'name']], on='id', how='left')
data = data.dropna()

In [40]:
# KNN color clusters
coords = data[['lat', 'lon']]
nbrs = NearestNeighbors(n_neighbors=10).fit(coords)
distances, indices = nbrs.kneighbors(coords)
group_labels = np.mean(distances, axis=1)  # use mean distance as cluster strength
data['color_group'] = pd.qcut(group_labels, 10, labels=False, duplicates='drop')

In [41]:
# Plotly ScatterGeo with checkboxes
fig = go.Figure()

# For filtering
all_names = sorted(data['name'].unique())
colors = px.colors.qualitative.Plotly

# Add scatter plot for each bird
for i, name in enumerate(all_names):
    subset = data[data['name'] == name]
    fig.add_trace(go.Scattergeo(
        lon=subset['lon'],
        lat=subset['lat'],
        mode='markers',
        marker=dict(size=6, color=subset['color_group'], colorscale=[[0, 'blue'], [0.5, 'green'], [1, 'red']]
, showscale=False),
        name=name,
        visible=False
    ))

In [42]:
# Add UI buttons
buttons = []
for i, name in enumerate(all_names):
    visibility = [False] * len(all_names)
    visibility[i] = True
    buttons.append(dict(label=name,
                        method='update',
                        args=[{'visible': visibility},
                            {'title': f'Locations for: {name}'}]))

In [43]:
for i, name in enumerate(all_names):
    subset = data[data['name'] == name]
    # Create ogg file path (you can change path formatting if needed)
    subset = subset.copy()
    subset['ogg_path'] = 'train_audio/' + subset['id'] + '/' + subset['filename']
    fig.add_trace(go.Scattergeo(
        lon=subset['lon'],
        lat=subset['lat'],
        mode='markers',
        marker=dict(
            size=6,
            color=subset['color_group'],
            colorscale=[[0, 'blue'], [0.5, 'green'], [1, 'red']],
            cmin=0,
            cmax=data['color_group'].max(),
            showscale=False
        ),
        name=name,
        visible=False,
        text=subset['ogg_path'],  # This shows on hover and on click
        hoverinfo='text'
    ))


In [44]:

# Layout
fig.update_layout(
    updatemenus=[dict(type='dropdown', showactive=True, buttons=buttons,
                    x=0.01, y=1.05)],
    title='Bird Species Locations - BirdCLEF 2025',
    geo=dict(projection_type='natural earth',
            showland=True, landcolor='rgb(243, 243, 243)',
            showcountries=True, countrycolor='rgb(204, 204, 204)')
)

fig.write_html(output_html)